# Create the NFL PPR Google Sheet in your Google account

Run **Runtime → Run all**. Google will ask you to sign in once.

That creates a spreadsheet **you own**, with:
- **One tab per year** (`2021` … `2025`) — top 100 PPR at QB, RB, WR, TE
- **All_seasons** — every player-season stacked for Pivot Tables
- **Dashboard** — PPR leaders, position averages, scoring-champion chart

Scoring is **PPR** (`fantasy_points_ppr`). When it finishes, the live Google Sheets URL is printed at the bottom.

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Signed in.")

In [ ]:
import urllib.request
from pathlib import Path

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

XLSX = Path("/tmp/nfl_ppr_last_five_years.xlsx")
URLS = [
    "https://raw.githubusercontent.com/ashwinkren/ashwinkren/cursor/nba-player-stats-e7f6/data/nfl_ppr_last_five_years.xlsx",
    "https://cdn.jsdelivr.net/gh/ashwinkren/ashwinkren@cursor/nba-player-stats-e7f6/data/nfl_ppr_last_five_years.xlsx",
]

last_err = None
for url in URLS:
    try:
        print("Downloading", url)
        urllib.request.urlretrieve(url, XLSX)
        break
    except Exception as exc:
        last_err = exc
        print("  failed:", exc)
else:
    raise RuntimeError(f"Could not download workbook: {last_err}")

print("Local file", XLSX, "size", XLSX.stat().st_size, "bytes")

creds, _ = google.auth.default()
drive = build("drive", "v3", credentials=creds)

media = MediaFileUpload(
    XLSX,
    mimetype="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    resumable=True,
)
created = (
    drive.files()
    .create(
        body={
            "name": "NFL PPR player stats — 2021 to 2025",
            "mimeType": "application/vnd.google-apps.spreadsheet",
        },
        media_body=media,
        fields="id,name,webViewLink",
        supportsAllDrives=True,
    )
    .execute()
)

file_id = created["id"]
sheet_url = created["webViewLink"]

drive.permissions().create(
    fileId=file_id,
    body={"type": "anyone", "role": "reader"},
    fields="id",
).execute()

print("\nGoogle Sheet is ready (you are the owner):")
print(sheet_url)